# Model A vs. Model B: Is the Gap Real?

You ran both models on your eval set. Model A scored 78%, Model B scored 75%.
Is that a real difference? With N=50, the 95% CI on that 3-point gap often spans zero.

This notebook walks through the complete statistical workflow:
from raw scores to a defensible conclusion.

## The Setup

We evaluate **two models on the _same_ prompt template with 50s** (a *paired* design).
Each item gets a binary score: 1 if the model's response passes, 0 if it fails.
This is a classic setup for ML benchmarks.

The paired structure is key: because each prompt is seen by both models,
we'll compute a per-item *difference* and bootstrap that directly.
This is more statistically powerful than treating the two samples as independent.

We'll use synthetic data here, but you can swap these scores with your own eval results.

In [1]:
import numpy as np
import promptstats as pstats

rng = np.random.default_rng(42)
N = 50

# Synthetic binary scores (1 = pass, 0 = fail)
# We rig it so that the true mean of Model A = 0.70 (performance is 70% accurate on these inputs),
# while Model B's true performance is 0.65. That's a 5-point gap.
# We use `rng.binomial` to simulate the binary outcomes for each model across N samples:
# we draw 0s and 1s with probabilities 0.70 and 0.65 for Models A and B, respectively.
scores_a = rng.binomial(1, 0.70, N).astype(float)
scores_b = rng.binomial(1, 0.65, N).astype(float)

mean_a = scores_a.mean()
mean_b = scores_b.mean()
obs_diff = mean_a - mean_b

print(f"Model A:  {mean_a:.1%}  ({int(scores_a.sum())}/{N} passed)")
print(f"Model B:  {mean_b:.1%}  ({int(scores_b.sum())}/{N} passed)")
print(f"Gap:     {obs_diff:+.1%}")

Model A:  64.0%  (32/50 passed)
Model B:  72.0%  (36/50 passed)
Gap:     -8.0%


## Computing a CI on the Difference

**What's a confidence interval?** A 95% CI is a range computed from your sample
such that, if you repeated the experiment many times, about 95% of those ranges
would contain the true value. Think of it like a poll's margin of error: a wider
CI means less certainty, a narrower CI means more. When the CI for a *gap*
includes zero, the data can't rule out that the true difference is zero.

The observed gap is a *sample statistic* — it estimates the true gap but carries
sampling uncertainty. We use `pstats.compare_models()` to get confidence
intervals on each model's mean *and* on the gap between them.

Internally, the function treats the aligned score arrays as a paired design (each row
is the same prompt seen by both models) and bootstraps per-item differences.
We use `method="auto"`, which looks at the setup and input data types and tries to 
deduce the most appropriate statistical test to run for the comparison.

In [2]:
report = pstats.compare_models(
    {"Model A": scores_a, "Model B": scores_b},
    method="auto",
    rng=np.random.default_rng(0),
)

# Per-model statistics
for label, stats in report.model_stats.items():
    print(f"{label}: mean={stats.mean:.1%}  95% CI [{stats.ci_low:.1%}, {stats.ci_high:.1%}]")

print()

# Gap between the two models: pairwise summary with ASCII interval plot and verdict
pair = report.pairwise.get("Model A", "Model B")
pair.summary()

Model A: mean=64.0%  95% CI [50.1%, 75.9%]
Model B: mean=72.0%  95% CI [58.3%, 82.5%]

 PAIRWISE: MODEL A VS. MODEL B 
  method: bayes binary (n=10000)  |  N=50 inputs

  Mean gap (Model A − Model B):  -8.0%
  95% CI:  [-24.1%, +9.0%]

  axis: [-0.714, +0.714]  (· ±1σ spread, ─ 95% CI, ● mean, │ zero)
  Model B (<0) ················──────●─│────················      (>0) Model A

  Effect size (rank-biserial r):  -0.200   p (Wilcoxon signed-rank) = 0.3711  (not significant)

  Verdict: CI spans zero — the gap is not distinguishable from chance at N=50 (Wilcoxon signed-rank p=0.371). Collect more data to be sure.



## What's going on here?

When you run a pairwise comparison, you're asking: "across my test inputs, which model tended to score higher, and by how much?" The **mean gap** is the answer—in this case, -8%, meaning Model B came out ahead by about 8 points on average. While that's a useful number, it's just a point estimate from 50 samples. On its own, it doesn't tell you how confident you should be that this gap reflects a real difference, rather than just the luck of which inputs you happened to test on.

That's where the **confidence interval (CI)** comes in, and it's really the most important thing to look at. The CI of `[-24.1%, +9.0%]` (the `····──────●─│────···` in the plot) is telling you: given your data, the true underlying gap between these models could plausibly be anywhere in that range. Model B could be a lot better, or Model A could actually be slightly better — both are consistent with what you observed. The CI is wide here, and straddles zero `|` which represents "no difference" between Model A and B. Why? N=50 is a fairly small sample, and the model distributions aren't that large. This isn't a failure; it's the eval being honest with you about how much you can conclude. The goal as you collect more data is to watch that interval *narrow*, and pay attention to whether and how it does. 

The **effect size** (`r = -0.200`) and **p-value** (`p = 0.371`) are also reported here, but don't lean on them too hard. The p-value in particular is easy to misread as a pass/fail signal. With only 50 inputs, a p-value above 0.05 just means your sample is too small to rule out noise, which the wide CI already told you. The effect size is more useful: it's a standardised way of saying the gap is small-to-moderate in magnitude, which is worth knowing when you're deciding whether the difference even matters for your use case. (`prompstats` uses a rank-biserial effect index by default as we can't assume LLM responses follow a normal distribution.)

Basically this plot tells us—we can't decide what model is better. We need to keep collecting eval inputs, and possibly expand our test set. If the 95% CI narrows and stabilizes such that it doesn't include 0—e.g., imagine the plot looks like:

```
Model B (<0)   ··───●───··      |              (>0) Model A
```

Then we might draw a firmer conclusion that Model B is reliably better for these inputs. 

_*Note that `bayes_binary` is named as the test. Since we set `method="auto"`, `promptstats` looked at our data and deduced it was binary (all 0 or 1s). Since we're comparing the exact same inputs to two different models (a "pairwise" comparison), `promptstats` uses the Bayesian pairwise method from `bayes_evals` (https://github.com/sambowyer/bayes_evals) for calculating confidence intervals, as our simulations suggest it is the most reliable statistical method for this data type and sample size. However, if you'd like to use another method, like Newcombe (the pairwise sibling test to the Wilson score interval), you can tell `analyze` to use that method (here `method="newcombe"`)._

## But wait, didn't we rig Model B to be _worse_ than Model A?!

Actually, yes, good point. That's why statistics are so important. If we didn't have statistics and were eyeballing a bar chart: 

```
Mean Accuracy
───────────────────────────────────────────
Model A  ████████████████████        64.0%
Model B  ██████████████████████████  72.0%
───────────────────────────────────────────
```

we might just conclude that Model B is better, and call it a day. After all, we have `N=50` samples, and that sounds like a lot. 

This is where confidence intervals save us. Because we see:

```
Mean gap (Model A − Model B):  -8.0%
95% CI:  [-24.1%, +9.0%]

  axis: [-0.714, +0.714]  (· ±1σ spread, ─ 95% CI, ● mean, │ zero)
  Model B (<0) ················──────●─│────················      (>0) Model A
```

and that 95% CI spans 0, we can't be sure what world we're in—one where Model B is truly better, or just bad luck of the draw. You wouldn't catch this, then you'd make a wrong choice, then users will suffer (in a minor way, but still one that will impact your bottom line).

## Improving the test: Prefer to have multiple runs

Actually, there's something we should've done better here. LLMs are stochastic, but we just ran 50 inputs a single time through these models. That's actually a bad idea. If we want to be more precise, we should at least add multiple runs to quantify how much that performance is due to chance—at least, say, `N=3`. 

Let's do that now—let's add multiple runs, and see what happens. 

## Interactive CI Visualizer

Run the next cell to see an animated explanation of how repeated finite samples produce confidence intervals for the gap $(A - B)$.

- The top panel shows sampled model scores and sample means.
- The bottom panel stacks the resulting confidence intervals.
- Green intervals cover the true effect; red intervals miss it.

In [32]:
from IPython.display import HTML, display
import html as _html

ci_widget_html = r'''<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>LLM Eval: From Samples to Confidence Intervals</title>
  <script src="https://d3js.org/d3.v7.min.js"></script>
  <style>
    html, body {
      background: #ffffff;
      color: #111111;
    }
    body { font-family: sans-serif; margin: 0; padding: 12px; }
    .curveA { stroke: #1f77b4; fill: none; }
    .curveB { stroke: #ff7f0e; fill: none; }
    .dotA { fill: #1f77b4; opacity: 0.6; }
    .dotB { fill: #ff7f0e; opacity: 0.6; }
    .meanA { stroke: #1f77b4; stroke-width: 2; }
    .meanB { stroke: #ff7f0e; stroke-width: 2; }
    .ci.hit { stroke: #2ca02c; }
    .ci.miss { stroke: #d62728; }
    .zero-line { stroke: black; stroke-dasharray: 4; }
    .true-line { stroke: #7c3aed; stroke-dasharray: 3; }
    .label { font-size: 12px; fill: #333; }
    button { margin-right: 8px; margin-top: 8px; }
  </style>
</head>
<body>

<label>True difference:
  <input type="range" id="effect" min="0" max="0.3" step="0.01" value="0.08">
  <span id="effectVal">0.05</span>
</label>

<br>

<label>Dataset size (n):
  <input type="range" id="n" min="10" max="200" step="10" value="50">
  <span id="nVal">50</span>
</label>

<br>

<label>Model A sharpness (sd):
  <input type="range" id="sdA" min="0.03" max="0.25" step="0.01" value="0.10">
  <span id="sdAVal">0.10</span>
</label>

<br>

<label>Model B sharpness (sd):
  <input type="range" id="sdB" min="0.03" max="0.25" step="0.01" value="0.10">
  <span id="sdBVal">0.10</span>
</label>

<br>

<button onclick="runAnimated()">Run 1 eval (animated)</button>
<button onclick="resetPlot()">Reset</button>

<svg id="top" width="800" height="250"></svg>
<svg id="bottom" width="800" height="250"></svg>

<p id="status"></p>

<script>
const width = 800;

const effectSlider = document.getElementById("effect");
const nSlider = document.getElementById("n");
const sdASlider = document.getElementById("sdA");
const sdBSlider = document.getElementById("sdB");

effectSlider.oninput = () => {
  document.getElementById("effectVal").innerText = effectSlider.value;
  drawDistributions();
};

nSlider.oninput = () => {
  document.getElementById("nVal").innerText = nSlider.value;
};

sdASlider.oninput = () => {
  document.getElementById("sdAVal").innerText = Number(sdASlider.value).toFixed(2);
  drawDistributions();
};

sdBSlider.oninput = () => {
  document.getElementById("sdBVal").innerText = Number(sdBSlider.value).toFixed(2);
  drawDistributions();
};

function getEffect() { return parseFloat(effectSlider.value); }
function getN() { return parseInt(nSlider.value); }
function getSdA() { return parseFloat(sdASlider.value); }
function getSdB() { return parseFloat(sdBSlider.value); }

const x = d3.scaleLinear().domain([0,1]).range([50,750]);
const y = d3.scaleLinear().domain([0,5]).range([200,20]);

const topSvg = d3.select("#top");
const bottomSvg = d3.select("#bottom");

function normalPDF(x, mean, sd) {
  return (1 / (sd * Math.sqrt(2*Math.PI))) *
         Math.exp(-0.5 * ((x - mean)/sd)**2);
}

function drawDistributions() {
  topSvg.selectAll("*").remove();

  const effect = getEffect();
  const base = 0.5;
  const meanA = base + effect/2;
  const meanB = base - effect/2;
  const sdA = getSdA();
  const sdB = getSdB();
  const peakYA = Math.max(12, y(normalPDF(meanA, meanA, sdA)) - 8);
  const peakYB = Math.max(12, y(normalPDF(meanB, meanB, sdB)) - 8);

  const data = d3.range(0,1,0.01);

  const lineA = d3.line()
    .x(d => x(d))
    .y(d => y(normalPDF(d, meanA, sdA)));

  const lineB = d3.line()
    .x(d => x(d))
    .y(d => y(normalPDF(d, meanB, sdB)));

  topSvg.append("path").datum(data).attr("class","curveA").attr("d", lineA);
  topSvg.append("path").datum(data).attr("class","curveB").attr("d", lineB);

  topSvg.append("text")
    .attr("class","label")
    .attr("x", 400)
    .attr("y", 240)
    .attr("text-anchor","middle")
    .text("Model score");

  topSvg.append("text")
    .attr("class","label")
    .attr("x", x(meanA))
    .attr("y", peakYA)
    .attr("text-anchor","middle")
    .text("Model A");

  topSvg.append("text")
    .attr("class","label")
    .attr("x", x(meanB))
    .attr("y", peakYB)
    .attr("text-anchor","middle")
    .text("Model B");
}

let intervals = [];

async function runAnimated() {
  const effect = getEffect();
  const n = getN();

  const base = 0.5;
  const sdA = getSdA();
  const sdB = getSdB();

  let samplesA = [];
  let samplesB = [];

  document.getElementById("status").innerText = "Sampling...";

  for (let i = 0; i < n; i++) {
    const a = d3.randomNormal(base + effect/2, sdA)();
    const b = d3.randomNormal(base - effect/2, sdB)();

    samplesA.push(a);
    samplesB.push(b);

    topSvg.append("circle")
      .attr("class","dotA")
      .attr("cx", x(a))
      .attr("cy", 210)
      .attr("r", 3);

    topSvg.append("circle")
      .attr("class","dotB")
      .attr("cx", x(b))
      .attr("cy", 220)
      .attr("r", 3);

    await new Promise(r => setTimeout(r, 15));
  }

  const meanA = d3.mean(samplesA);
  const meanB = d3.mean(samplesB);
  const diff = meanA - meanB;

  topSvg.append("line")
    .attr("class","meanA")
    .attr("x1", x(meanA))
    .attr("x2", x(meanA))
    .attr("y1", 0)
    .attr("y2", 200);

  topSvg.append("line")
    .attr("class","meanB")
    .attr("x1", x(meanB))
    .attr("x2", x(meanB))
    .attr("y1", 0)
    .attr("y2", 200);

  document.getElementById("status").innerText =
    `Sample diff (A − B) = ${diff.toFixed(3)}`;

  const se = Math.sqrt((sdA**2)/n + (sdB**2)/n);
  const margin = 1.96 * se;
  const ci = [diff - margin, diff + margin];

  const contains = ci[0] <= effect && ci[1] >= effect;
  intervals.push({ci, contains, diff});

  drawCI();
}

function drawCI() {
  bottomSvg.selectAll("*").remove();

  const trueEffect = getEffect();
  const xCI = d3.scaleLinear().domain([-0.3,0.3]).range([50,750]);

  bottomSvg.append("text")
    .attr("class","label")
    .attr("x", 400)
    .attr("y", 234)
    .attr("text-anchor","middle")
    .text("Performance difference (A − B)");

  bottomSvg.append("line")
    .attr("class","zero-line")
    .attr("x1", xCI(0))
    .attr("x2", xCI(0))
    .attr("y1", 0)
    .attr("y2", 250);

  bottomSvg.append("line")
    .attr("class","true-line")
    .attr("x1", xCI(trueEffect))
    .attr("x2", xCI(trueEffect))
    .attr("y1", 0)
    .attr("y2", 250);

  bottomSvg.append("text")
    .attr("class","label")
    .attr("x", xCI(trueEffect))
    .attr("y", 250)
    .attr("text-anchor","middle")
    .text("True mean difference");

  bottomSvg.selectAll(".ci")
    .data(intervals)
    .enter()
    .append("line")
    .attr("class", d => "ci " + (d.contains ? "hit" : "miss"))
    .attr("x1", d => xCI(d.ci[0]))
    .attr("x2", d => xCI(d.ci[1]))
    .attr("y1", (d,i) => 20 + i*6)
    .attr("y2", (d,i) => 20 + i*6)
    .attr("stroke-width", 2);

  const last = intervals[intervals.length - 1];
  if (last) {
    bottomSvg.append("text")
      .attr("class","label")
      .attr("x", xCI(last.ci[1]) + 5)
      .attr("y", 24 + (intervals.length - 1)*6)
      .text(`95% CI [${last.ci[0].toFixed(3)}, ${last.ci[1].toFixed(3)}]`);
  }
}

function resetPlot() {
  intervals = [];
  topSvg.selectAll("*").remove();
  bottomSvg.selectAll("*").remove();
  drawDistributions();
  document.getElementById("status").innerText = "";
}

drawDistributions();
</script>

</body>
</html>'''

iframe_srcdoc = _html.escape(ci_widget_html)

display(HTML(f'''
<iframe
  title="Confidence interval visualizer"
  style="width: 100%; max-width: 860px; height: 620px; border: 1px solid #d1d5db; border-radius: 8px;"
  sandbox="allow-scripts"
  srcdoc="{iframe_srcdoc}">
</iframe>
'''))

## What 'Statistically Distinguishable' Means

A CI that spans zero does **not** mean the two models are equal.
It means your data is **insufficient to rule out** the possibility that they are.
The gap could be real — you just don't have enough samples to confirm it.

The honest report is:
> *"Model A scored X% vs. Model B's Y% on our 50-item eval.
  The 95% CI on the gap is [lo, hi], which includes zero.
  We cannot conclude that Model A is better at this sample size."*

The wrong report is: *"Model A outperformed Model B (78% vs 75%)"* — full stop,
no uncertainty, no sample size, implying a definitive conclusion.

## How Much Data Do You Actually Need?

If the CI spans zero, the next question is: *how large would N need to be*
to reliably detect the gap you're seeing?

We estimate this with a simple simulation: for a range of N values,
we repeatedly sample from models with the observed means and check what
fraction of CIs exclude zero (= empirical power).

In [14]:
def empirical_power(p_a, p_b, n, n_sims=100, alpha=0.05, seed=1):
    """Fraction of simulated trials where the gap is declared significant."""
    rng_sim = np.random.default_rng(seed)
    detected = 0
    for _ in range(n_sims):
        a = rng_sim.binomial(1, p_a, n).astype(float)
        b = rng_sim.binomial(1, p_b, n).astype(float)
        rep = pstats.compare_models(
            {"Model A": a, "Model B": b},
            method="auto",
            alpha=alpha,
            rng=rng_sim,
        )
        if rep.significant:
            detected += 1
    return detected / n_sims


# True pass rates from our synthetic setup
p_a, p_b = 0.70, 0.65  # 5pp true gap

print(f"Power to detect {p_a - p_b:.0%} gap (binary scores, 95% CI):\n")
for n in [50, 100, 200, 400]:
    power = empirical_power(p_a, p_b, n)
    bar = "█" * int(power * 30)
    print(f"  N={n:4d}  {power:.0%}  {bar}")

Power to detect 5% gap (binary scores, 95% CI):

  N=  50  7%  ██
  N= 100  13%  ███
  N= 200  16%  ████
  N= 400  29%  ████████


## Summary

| What you have | What you can say |
|---|---|
| CI excludes zero, both bounds positive | Model A is reliably better |
| CI spans zero | Too close to call at this N |
| CI excludes zero, both bounds negative | Model B is reliably better |

**Key takeaways:**

- For binary scores, detecting a 5pp gap requires **N ≈ 200–400** per model at 80–95% power.
  Most evals use N=50–100, which can only reliably detect gaps of 10–15pp or larger.
- A paired design (same prompts for both models) is more powerful than independent samples.
  Use it whenever possible.
- Report CIs, not just means. A mean difference without a CI is an incomplete result.

Next: [Finding Your Best Prompt](./best-prompt.html) — extends this to k>2 comparisons
with multiple-comparison correction.